<a href="https://colab.research.google.com/github/lenjulca100/SCRIPT-VARIOS/blob/main/MODELO_PROYECTADO_NDVI_CAJAMARCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELDA 1: Instalación de paquetes
# -----------------------------------
print("Instalando dependencias de GEE y geoespaciales...")

!pip install earthengine-api -q
!pip install geemap -q
!pip install pyproj -q

print("¡Dependencias listas!")

Instalando dependencias de GEE y geoespaciales...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.8 MB/s eta 0:00:00
¡Dependencias listas!


In [3]:
# 2. Inicializar Earth Engine
# ---------------------------
def init_ee():
    try:
        ee.Initialize()
        print("✅ Earth Engine inicializado")
        return True
    except Exception as ex:
        print("🔐 Earth Engine no inicializado. Intentando autenticación interactiva...")
        try:
            ee.Authenticate()
            ee.Initialize(project='ee-cajamarca01')
            print("✅ Autenticación e inicialización completadas")
            return True
        except Exception as e2:
            print("❌ Error inicializando Earth Engine:", e2)
            return False

GEE_OK = init_ee()

🔐 Earth Engine no inicializado. Intentando autenticación interactiva...
✅ Autenticación e inicialización completadas


In [8]:
# CELDA 3: Script Completo (Configuración, Funciones y Ejecución)
# -----------------------------------------------------------------
# (VERSIÓN 2.0 - Arregla el R2 bajo y usa datos desde 1985)

# 1. Importaciones
import os
import math
import time
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyproj
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from joblib import dump
from datetime import datetime
from matplotlib.colors import LinearSegmentedColormap

# Configuraciones de gráficos
sns.set(style="whitegrid")
plt.rcParams.update({'figure.max_open_warning': 0})

# ---------------------------
# 0. Parámetros generales
# ---------------------------
# --- ¡CAMBIO IMPORTANTE! ---
START_YEAR = 1985
END_YEAR = 2024
PRED_START_YEAR = 2025
PRED_END_YEAR = 2050

# --- ¡CAMBIO IMPORTANTE! ---
# Usamos AVHRR (desde 1981) en lugar de MODIS (desde 2000)
AVHRR_PRODUCT = 'NOAA/CDR/AVHRR/NDVI/V5'
AVHRR_SCALE = 5000 # La escala de AVHRR es ~5.6km, 5000 es apropiado
CHIRPS_PRODUCT = 'UCSB-CHG/CHIRPS/DAILY'
CHIRPS_SCALE = 5000
ERA5_PRODUCT = 'ECMWF/ERA5_LAND/HOURLY'
ERA5_SCALE = 10000 # ERA5-Land es ~11km

GEE_PROJECT_ID = 'ee-cajamarca01'

OUT_DIR = "ndvi_results_v2"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------
# 1. Coordenadas UTM (tus polígonos)
# ---------------------------
huanico_utm = [
    (808982.07, 9210637.96), (809848.13, 9206637.20), (812644.89, 9205422.84),
    (815792.64, 9205458.25), (817838.86, 9207588.18), (817280.23, 9211667.44),
    (815219.85, 9214168.00), (811169.56, 9214354.96), (808982.07, 9210637.96)
]
michiquillay_utm = [
    (798678.93, 9225523.12), (791832.43, 9221923.74), (792099.57, 9219173.28),
    (794509.47, 9216855.24), (798706.36, 9216305.42), (802389.71, 9217698.86),
    (804718.29, 9222172.80), (802476.73, 9225662.84), (798678.93, 9225523.12)
]

def utm_list_to_lonlat(poly_utm, zone=17, hemisphere='S'):
    crs_latlon = "EPSG:4326"
    epsg_utm = 32700 + int(zone) if hemisphere.upper() == 'S' else 32600 + int(zone)
    crs_utm = f"EPSG:{epsg_utm}"
    transformer = pyproj.Transformer.from_crs(crs_utm, crs_latlon, always_xy=True)
    out = []
    for e, n in poly_utm:
        lon, lat = transformer.transform(e, n)
        out.append([lon, lat])
    return out

poly_huanico = utm_list_to_lonlat(huanico_utm)
poly_mich = utm_list_to_lonlat(michiquillay_utm)
AREAS = {
    "Huanico": ee.Geometry.Polygon(poly_huanico),
    "Michiquillay": ee.Geometry.Polygon(poly_mich)
}

# ---------------------------
# 2. Inicializar Earth Engine (Corregido)
# ---------------------------
def init_ee():
    try:
        ee.Initialize(project=GEE_PROJECT_ID)
        print(f"✅ Earth Engine inicializado con proyecto: {GEE_PROJECT_ID}")
        return True
    except Exception as ex:
        print(f"❌ Error fatal inicializando Earth Engine: {ex}")
        return False

GEE_OK = init_ee()

# ---------------------------
# 3. Funciones de extracción (monthly)
# ---------------------------
# --- ¡CAMBIO IMPORTANTE! ---
# Renombrada y adaptada para AVHRR
def get_monthly_ndvi(year, month, geom):
    """
    Retorna promedio NDVI (escalado) para el mes usando AVHRR.
    """
    try:
        start = ee.Date.fromYMD(int(year), int(month), 1)
        end = start.advance(1, 'month')
        col = (ee.ImageCollection(AVHRR_PRODUCT)
               .filterDate(start, end)
               .select('NDVI')) # La banda se llama 'NDVI'

        # El factor de escala de AVHRR V5 es 0.0001
        def scale(img):
            return img.multiply(0.0001).rename('NDVI').copyProperties(img, ['system:time_start'])

        col = col.map(scale)
        # Usamos la mediana para un mes de datos diarios
        med = col.median().clip(geom)
        stat = med.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=AVHRR_SCALE,
            maxPixels=1e9
        )
        return stat.get('NDVI').getInfo()
    except Exception as e:
        return None

def chirps_monthly_precip(year, month, geom):
    try:
        start = ee.Date.fromYMD(int(year), int(month), 1)
        end = start.advance(1, 'month')
        col = ee.ImageCollection(CHIRPS_PRODUCT).filterDate(start, end)
        summed = col.sum().clip(geom)
        stat = summed.reduceRegion(ee.Reducer.mean(), geom, scale=CHIRPS_SCALE, maxPixels=1e9)
        return stat.get('precipitation').getInfo()
    except Exception:
        return None

def era5_monthly_temp(year, month, geom):
    try:
        start = ee.Date.fromYMD(int(year), int(month), 1)
        end = start.advance(1, 'month')
        col = ee.ImageCollection(ERA5_PRODUCT).filterDate(start, end).select('temperature_2m')
        avg = col.mean().clip(geom)
        stat = avg.reduceRegion(ee.Reducer.mean(), geom, scale=ERA5_SCALE, maxPixels=1e9)
        temp_k = stat.get('temperature_2m').getInfo()
        if temp_k is None:
            return None
        return temp_k - 273.15 # Convertir a Celsius
    except Exception:
        return None

# ---------------------------
# 4. Construir dataset histórico mensual (por área)
# ---------------------------
def build_monthly_df_for_area(area_name, geom, start_year=START_YEAR, end_year=END_YEAR):
    rows = []
    # Generar la cuadrícula de fechas
    dates = pd.date_range(start=f'{start_year}-01-01', end=f'{end_year}-12-01', freq='MS')

    print(f"Iniciando extracción para {area_name} desde {start_year} hasta {end_year}...")

    for dt in dates:
        y, m = dt.year, dt.month
        print(f"Extrayendo {area_name} {y}-{m:02d} ...", end='\r')

        # --- ¡CAMBIO IMPORTANTE! ---
        # Usamos la nueva función get_monthly_ndvi
        ndvi = get_monthly_ndvi(y, m, geom) if GEE_OK else None
        precip = chirps_monthly_precip(y, m, geom) if GEE_OK else None
        temp = era5_monthly_temp(y, m, geom) if GEE_OK else None

        rows.append({'year': y, 'month': m, 'ndvi': ndvi, 'precip': precip, 'temp': temp})

    print(f"\nExtracción completa para {area_name}. Rellenando huecos...")
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
    df = df.set_index('date')

    # Interpolación
    df = df.interpolate(method='time', limit_direction='both')

    # Lógica de llenado robusta (igual que antes, muy buena)
    defaults = {'ndvi': 0.5, 'precip': 50.0, 'temp': 15.0}
    for col in ['ndvi', 'precip', 'temp']:
        if df[col].isnull().all():
            print(f" ❌ CRÍTICO: Toda la columna '{col}' es NaN para {area_name}. Usando default {defaults[col]}.")
            df[col] = df[col].fillna(defaults[col])
            continue
        if df[col].isnull().any():
            print(f" ⚠️ Warning: NaNs restantes en '{col}' para {area_name}. Rellenando con climatología mensual.")
            monthly_climatology = df.groupby(df.index.month)[col].transform('mean')
            df[col] = df[col].fillna(monthly_climatology)
            if df[col].isnull().any():
                print(f" ⚠️ Info: Rellenando NaNs restantes en '{col}' con media global.")
                global_mean = df[col].mean()
                df[col] = df[col].fillna(global_mean if not pd.isna(global_mean) else defaults[col])

    df = df.reset_index(drop=False)
    return df

# ---------------------------
# 5. Modelado y predicción iterativa mes-a-mes
# ---------------------------

# --- ¡CAMBIO CLAVE PARA EL R2! ---
def prepare_features(df):
    """
    Función MEJORADA (V2):
    - Añade lags de clima (la lluvia de los meses pasados)
    - Añade una suma de precipitación de 3 meses (las plantas aman esto)
    - Añade más lags de NDVI
    """
    df2 = df.copy().set_index('date')

    # Lags del NDVI (memoria de la planta)
    df2['ndvi_lag1'] = df2['ndvi'].shift(1)
    df2['ndvi_lag3'] = df2['ndvi'].shift(3)
    df2['ndvi_lag12'] = df2['ndvi'].shift(12)

    # Lags de Clima (memoria del ambiente)
    df2['precip_lag1'] = df2['precip'].shift(1)
    df2['precip_lag3'] = df2['precip'].shift(3)
    df2['temp_lag1'] = df2['temp'].shift(1)

    # Precipitación acumulada (el "tanque de agua" del suelo)
    df2['precip_roll_3m'] = df2['precip'].rolling(window=3, min_periods=1).sum()
    df2['precip_roll_6m'] = df2['precip'].rolling(window=6, min_periods=1).sum()

    # El 'time' sigue siendo útil para tendencias
    df2 = df2.reset_index()
    df2['time'] = (df2['year'] - df2['year'].min()) * 12 + (df2['month'] - 1)

    # Borramos los NaNs creados por los lags (especialmente ndvi_lag12)
    df2 = df2.dropna().reset_index(drop=True)

    # ¡IMPORTANTE! Actualizamos la lista de features
    features = [
        'year', 'month', 'precip', 'temp', 'time',
        'ndvi_lag1', 'ndvi_lag3', 'ndvi_lag12',
        'precip_lag1', 'precip_lag3', 'temp_lag1',
        'precip_roll_3m', 'precip_roll_6m'
    ]

    # Asegurarnos de que solo usamos los features que existen en df2
    final_features = [f for f in features if f in df2.columns]

    return df2, final_features

def train_model(df_features, features, target='ndvi'):
    X = df_features[features]
    y = df_features[target]

    if X.empty:
        print("❌ Error: El DataFrame de features está vacío después de crear lags.")
        return None, None, None, None, None, None, None

    # Usar TimeSeriesSplit para una validación cruzada temporal más robusta
    tscv = TimeSeriesSplit(n_splits=5)

    # Entrenar el modelo final en ~80% de los datos, testear en el último ~20%
    split_index = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

    if X_train.empty or X_test.empty:
        print("❌ Error: No hay suficientes datos para dividir en train/test.")
        return None, None, None, None, None, None, None

    # Modelo más robusto: min_samples_leaf ayuda a prevenir overfitting
    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=5, # ¡NUEVO!
        max_features=0.5    # ¡NUEVO!
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'mae': mean_absolute_error(y_test, y_pred),
        'r2': r2_score(y_test, y_pred)
    }
    return model, X_train, X_test, y_train, y_test, y_pred, metrics

# --- ¡FUNCIÓN CLAVE REESCRITA! ---
def iterative_forecast_complete(model, hist_df_with_features, features,
                                start_year=PRED_START_YEAR, end_year=PRED_END_YEAR,
                                precip_pct_per_decade=0.0, temp_increase_per_decade=0.5):
    """
    Pronóstico iterativo V2:
    - Mantiene un historial rodante de features.
    - Calcula los nuevos lags (precip_lag1, ndvi_lag3, precip_roll_3m, etc.) en cada paso.
    """

    # Usamos hist_df_with_features (el que sale de prepare_features)
    # como nuestra base de historial.
    # Necesitamos el hist_df original (sin 'dropna') para la climatología

    print("Iniciando pronóstico iterativo con features complejos...")

    # 1. Preparar climatología base
    last_period_start = max(hist_df_with_features['year'].max() - 9, hist_df_with_features['year'].min()) # 10 años
    last_period = hist_df_with_features[hist_df_with_features['year'] >= last_period_start]
    clim = last_period.groupby('month')[['precip', 'temp']].mean()

    # 2. Preparar el historial rodante
    # Necesitamos un DataFrame que se actualice con CADA predicción
    # Usamos el hist_df original (de build_monthly_df) como base
    # porque 'prepare_features' elimina filas con NaNs

    # Usamos 'date', 'year', 'month', 'ndvi', 'precip', 'temp'
    # El 'hist_df' que entra a esta función debe ser el 'df_hist' de 'run_for_area'
    # (El que tiene todos los años, sin 'dropna')
    history_df = hist_df_with_features.copy()

    forecast_rows = []
    min_hist_year = hist_df_with_features['year'].min()

    # Fechas de pronóstico
    pred_dates = pd.date_range(start=f'{start_year}-01-01', end=f'{end_year}-12-01', freq='MS')

    for dt in pred_dates:
        year, month = dt.year, dt.month

        # 1. Obtener fila actual de historial (la última)
        # Esto es lo que usaremos para calcular los lags
        last_known_row = history_df.iloc[-1]

        # 2. Aplicar escenario climático
        years_ahead = year - END_YEAR
        clim_row = clim.loc[month]
        precip_base = clim_row['precip']
        temp_base = clim_row['temp']

        precip_adj = max(0, precip_base * (1.0 + (precip_pct_per_decade / 10.0) * (years_ahead / 10.0)))
        temp_adj = temp_base + temp_increase_per_decade * (years_ahead / 10.0)

        # 3. Construir vector de features (LA PARTE DIFÍCIL)
        # Necesitamos calcular todos los lags y rolling basándonos en 'history_df'
        row_feat = {'year': year, 'month': month, 'precip': precip_adj, 'temp': temp_adj}
        row_feat['time'] = (year - min_hist_year) * 12 + (month - 1)

        # ndvi lags
        row_feat['ndvi_lag1'] = history_df['ndvi'].iloc[-1]
        row_feat['ndvi_lag3'] = history_df['ndvi'].iloc[-3]
        row_feat['ndvi_lag12'] = history_df['ndvi'].iloc[-12]

        # precip lags
        row_feat['precip_lag1'] = history_df['precip'].iloc[-1]
        row_feat['precip_lag3'] = history_df['precip'].iloc[-3]

        # temp lags
        row_feat['temp_lag1'] = history_df['temp'].iloc[-1]

        # rolling sums (más complejo)
        # Tomamos los últimos N-1 valores del historial y añadimos el actual
        precip_last_3 = list(history_df['precip'].iloc[-2:]) + [precip_adj]
        row_feat['precip_roll_3m'] = sum(precip_last_3)

        precip_last_6 = list(history_df['precip'].iloc[-5:]) + [precip_adj]
        row_feat['precip_roll_6m'] = sum(precip_last_6)

        # 4. Predecir
        # Asegurarse de que el orden de las columnas es el mismo
        Xrow = pd.DataFrame([row_feat])[features]

        pred_ndvi = model.predict(Xrow)[0]
        pred_ndvi = float(np.clip(pred_ndvi, 0.0, 1.0)) # Clamp

        # 5. Guardar resultado
        forecast_rows.append({
            'date': dt, 'year': year, 'month': month,
            'pred_ndvi': pred_ndvi,
            'precip': precip_adj, 'temp': temp_adj
        })

        # 6. --- ¡CRUCIAL! ---
        # Actualizar el historial rodante con la nueva predicción
        new_history_row = {
            'date': dt, 'year': year, 'month': month,
            'ndvi': pred_ndvi, # ¡Usamos el valor predicho!
            'precip': precip_adj,
            'temp': temp_adj
        }
        history_df = pd.concat([history_df, pd.DataFrame([new_history_row])], ignore_index=True)

    df_fore = pd.DataFrame(forecast_rows)
    return df_fore

# ---------------------------
# 6. Visualizaciones (MÁS GRÁFICOS)
# ---------------------------

# Gráfico 1: El Clásico
def plot_time_series_annual(df_hist, df_fore, area_name):
    hist_annual = df_hist.groupby('year')['ndvi'].mean().reset_index()
    fore_annual = df_fore.groupby('year')['pred_ndvi'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(15, 7))

    # Usar regplot para mostrar tendencias
    sns.regplot(x='year', y='ndvi', data=hist_annual, ax=ax, label='Histórico (Media Anual)', scatter_kws={'s': 30})
    sns.regplot(x='year', y='pred_ndvi', data=fore_annual, ax=ax, label=f'Proyección {PRED_START_YEAR}-{PRED_END_YEAR}',
                scatter_kws={'s': 30, 'alpha': 0.5}, line_kws={'linestyle': '--'})

    ax.set_title(f'NDVI Anual (con Tendencias): Histórico y Proyección - {area_name}')
    ax.set_xlabel('Año'); ax.set_ylabel('NDVI'); ax.set_ylim(0, 1)
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{area_name}_ndvi_annual_trend.png"), dpi=200)
    plt.close(fig)

# Gráfico 2: El Clásico Mensual
def plot_monthly_climatology(df_hist, df_fore, area_name):
    hist_month = df_hist.groupby('month')['ndvi'].agg(['mean', 'std']).reset_index()
    fore_month = df_fore.groupby('month')['pred_ndvi'].agg(['mean', 'std']).reset_index()

    fig, ax = plt.subplots(figsize=(12, 7))

    # Histórico con banda de error (1 std)
    ax.plot(hist_month['month'], hist_month['mean'], 'o-', label='Climatología Histórica (Media)')
    ax.fill_between(hist_month['month'], hist_month['mean'] - hist_month['std'], hist_month['mean'] + hist_month['std'],
                    alpha=0.2, label='Histórico (±1 std. dev.)')

    # Pronóstico con banda de error (1 std)
    ax.plot(fore_month['month'], fore_month['mean'], 's--', label='Climatología Proyectada (Media)')
    ax.fill_between(fore_month['month'], fore_month['mean'] - fore_month['std'], fore_month['mean'] + fore_month['std'],
                    color='red', alpha=0.2, label='Proyectado (±1 std. dev.)')

    ax.set_title(f'Climatología Mensual (Estacionalidad) - {area_name}')
    ax.set_xlabel('Mes'); ax.set_ylabel('NDVI'); ax.set_xticks(range(1, 13))
    ax.legend(); ax.grid()

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{area_name}_monthly_climatology.png"), dpi=200)
    plt.close(fig)

# Gráfico 3: Importancia (como antes)
def plot_feature_importance(model, features, area_name):
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1]

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(x=importances[idx], y=np.array(features)[idx], ax=ax)
    ax.set_title(f'Importancia de Variables (Modelo Mejorado) - {area_name}')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{area_name}_feat_importance.png"), dpi=200)
    plt.close(fig)

# Gráfico 4: Residuales (como antes)
def plot_residuals(y_test, y_pred, area_name):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(x=y_test, y=y_pred - y_test, alpha=0.6, ax=ax)
    ax.axhline(0, color='k', linestyle='--')
    ax.set_xlabel('NDVI Observado (Test)'); ax.set_ylabel('Residual (Pred - Obs)')
    ax.set_title(f'Residuales del Modelo - {area_name}')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{area_name}_residuals.png"), dpi=200)
    plt.close(fig)

# --- ¡NUEVO GRÁFICO! ---
# Gráfico 5: Boxplots por 5 Años
def plot_forecast_boxplots(df_fore, area_name):
    """
    Genera un boxplot que muestra la distribución del NDVI
    en períodos de 5 años.
    """
    # Crear los períodos de 5 años
    bins = [2024, 2030, 2035, 2040, 2045, 2050]
    labels = ["2025-2030", "2031-2035", "2036-2040", "2041-2045", "2046-2050"]
    df_fore['period'] = pd.cut(df_fore['year'], bins=bins, labels=labels, right=True)

    df_plot = df_fore.dropna(subset=['period'])

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.boxplot(x='period', y='pred_ndvi', data=df_plot, ax=ax)
    ax.set_title(f'Distribución del NDVI Proyectado (por 5 Años) - {area_name}')
    ax.set_xlabel('Período de Pronóstico')
    ax.set_ylabel('NDVI Proyectado')
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{area_name}_forecast_boxplots.png"), dpi=200)
    plt.close(fig)

# --- ¡NUEVO GRÁFICO! ---
# Gráfico 6: Mapa de Calor
def plot_climatology_heatmap(df, area_name, title_suffix="Histórico"):
    """
    Genera un mapa de calor Mes vs Año del NDVI.
    """
    if 'pred_ndvi' in df.columns:
        df['ndvi'] = df['pred_ndvi'] # Usar datos de predicción

    df_pivot = df.pivot_table(index='month', columns='year', values='ndvi', aggfunc='mean')

    fig, ax = plt.subplots(figsize=(20, 8))
    # Usar un colormap "verde"
    cmap = LinearSegmentedColormap.from_list("green_ndvi", ["white", "lightgreen", "green", "darkgreen"])

    sns.heatmap(df_pivot, ax=ax, cmap=cmap, vmin=0, vmax=1, annot=False)
    ax.set_title(f'Mapa de Calor NDVI (Mes vs. Año) - {area_name} ({title_suffix})')
    ax.set_xlabel('Año')
    ax.set_ylabel('Mes')

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{area_name}_heatmap_{title_suffix.lower()}.png"), dpi=200)
    plt.close(fig)

# ---------------------------
# 7. Estadísticas y Flujo Principal
# ---------------------------

# --- ¡NUEVA FUNCIÓN DE ESTADÍSTICAS! ---
def generate_summary_statistics(df_fore, area_name, scenario_name):
    """
    Calcula y guarda estadísticas descriptivas para cada
    período de 5 años en el pronóstico.
    """
    print(f"--- Estadísticas de 5 Años ({area_name} - {scenario_name}) ---")
    bins = [2024, 2030, 2035, 2040, 2045, 2050]
    labels = ["2025-2030", "2031-2035", "2036-2040", "2041-2045", "2046-2050"]
    df_fore['period'] = pd.cut(df_fore['year'], bins=bins, labels=labels, right=True)

    stats = df_fore.groupby('period')['pred_ndvi'].describe().round(4)

    print(stats.to_string()) # Imprimir en consola

    # Guardar en CSV
    stats_file = os.path.join(OUT_DIR, f"{area_name}_{scenario_name}_stats_5_anos.csv")
    stats.to_csv(stats_file)
    print(f"Estadísticas guardadas en: {stats_file}\n")
    return stats


def run_for_area(area_name, geom):
    print("\n" + "="*60)
    print(f"INICIANDO ANÁLISIS MEJORADO PARA: {area_name}")
    print(f"(Datos desde {START_YEAR} hasta {END_YEAR})")
    print("="*60)

    hist_csv_file = os.path.join(OUT_DIR, f"{area_name}_monthly_hist_{START_YEAR}-{END_YEAR}.csv")

    # --- Checkpoint: No volver a descargar si ya lo tenemos ---
    if os.path.exists(hist_csv_file):
        print(f"Cargando histórico existente desde: {hist_csv_file}")
        df_hist = pd.read_csv(hist_csv_file, parse_dates=['date'])
    else:
        df_hist = build_monthly_df_for_area(area_name, geom, START_YEAR, END_YEAR)
        df_hist.to_csv(hist_csv_file, index=False)
        print(f"\nHistórico construido: {df_hist.shape[0]} filas. Guardado CSV.")

    # preparar features MEJORADOS
    df_feat, features = prepare_features(df_hist.copy())

    if df_feat.empty:
        print(f"❌ CRÍTICO: No se pudieron generar features para {area_name}.")
        return None, None, None, None

    print("Features (con lags) creados. Primeras filas:")
    print(df_feat.head())
    print(f"Usando {len(features)} features: {features}")

    # Entrenar modelo
    model, X_train, X_test, y_train, y_test, y_pred, metrics = train_model(df_feat, features)
    if model is None:
        print(f"❌ CRÍTICO: Falló el entrenamiento del modelo para {area_name}.")
        return None, None, None, None

    print(f"\n--- ¡NUEVAS MÉTRICAS (Modelo V2)! ---")
    print(f"Modelo entrenado. Métricas (test): RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, R2={metrics['r2']:.4f}")
    print("------------------------------------------")

    # gráficos de validación
    plot_feature_importance(model, features, area_name)
    plot_residuals(y_test, y_pred, area_name)
    plot_climatology_heatmap(df_hist, area_name, "Historico") # Mapa de calor histórico

    # Escenarios:
    scenarios = [
        {'name': 'NoChange', 'precip_pct_decade': 0.0, 'temp_inc_decade': 0.0},
        {'name': 'DryWarmer', 'precip_pct_decade': -5.0, 'temp_inc_decade': 0.5},
        {'name': 'WetterWarmer', 'precip_pct_decade': 2.0, 'temp_inc_decade': 1.0}
    ]

    forecasts = {}
    for sc in scenarios:
        sc_name = sc['name']
        print(f"\nGenerando forecast escenario: {sc_name}")

        # --- ¡CAMBIO IMPORTANTE! ---
        # Pasamos df_feat (con features) para el historial rodante
        # Y df_hist (original) para la climatología
        # ... Corrección: la V2 de iterative_forecast ahora usa df_feat como 'history_df'
        # y el df_hist original (de 'build_...()') para la climatología.

        df_fore = iterative_forecast_complete(
            model, df_feat.copy(), features, # Pasamos df_feat como base del historial
            start_year=PRED_START_YEAR, end_year=PRED_END_YEAR,
            precip_pct_per_decade=sc['precip_pct_decade'],
            temp_increase_per_decade=sc['temp_inc_decade']
        )

        df_fore.to_csv(os.path.join(OUT_DIR, f"{area_name}_forecast_{sc_name}.csv"), index=False)
        forecasts[sc_name] = df_fore

        # Generar TODOS los gráficos y estadísticas
        area_sc_name = f"{area_name}_{sc_name}"
        plot_time_series_annual(df_hist, df_fore, area_sc_name)
        plot_monthly_climatology(df_hist, df_fore, area_sc_name)
        plot_forecast_boxplots(df_fore, area_sc_name)
        plot_climatology_heatmap(df_fore, area_sc_name, f"Forecast {sc_name}")

        # Generar la tabla de estadísticas
        generate_summary_statistics(df_fore.copy(), area_name, sc_name)

    # Guardar modelo
    dump(model, os.path.join(OUT_DIR, f"{area_name}_rf_model_v2.joblib"))
    print(f"\nModelo y pronósticos (V2) almacenados en {OUT_DIR}")

    return df_hist, model, metrics, forecasts

# ---------------------------
# 8. Ejecutar (sin modo sintético esta vez)
# ---------------------------
if __name__ == "__main__":
    if not GEE_OK:
        print("\n" + "!"*60)
        print("ATENCIÓN: Earth Engine NO se inicializó correctamente.")
        print("Revisa tu GEE_PROJECT_ID o la autenticación en la Celda 2.")
        print("!"*60 + "\n")
    else:
        # Ejecución real con GEE
        print("\nINICIANDO EJECUCIÓN REAL (V2) CON GOOGLE EARTH ENGINE...")
        results = {}
        for name, geom in AREAS.items():
            run_output = run_for_area(name, geom)
            if run_output and run_output[0] is not None:
                 results[name] = {
                     'hist': run_output[0],
                     'model': run_output[1],
                     'metrics': run_output[2],
                     'forecasts': run_output[3]
                }

    print(f"\nProceso finalizado. Revisa la carpeta '{OUT_DIR}' para CSVs, modelos y PNGs.")
    print("Usa el ícono de la carpeta a la izquierda para ver tus archivos.")

✅ Earth Engine inicializado con proyecto: ee-cajamarca01

INICIANDO EJECUCIÓN REAL (V2) CON GOOGLE EARTH ENGINE...

INICIANDO ANÁLISIS MEJORADO PARA: Huanico
(Datos desde 1985 hasta 2024)
Cargando histórico existente desde: ndvi_results_v2/Huanico_monthly_hist_1985-2024.csv
Features (con lags) creados. Primeras filas:
        date  year  month      ndvi      precip      temp  ndvi_lag1  \
0 1986-01-01  1986      1  0.161278   80.930177  8.422828   0.148256   
1 1986-02-01  1986      2  0.128819   95.029350  8.066077   0.161278   
2 1986-03-01  1986      3  0.154010  122.856670  8.108386   0.128819   
3 1986-04-01  1986      4  0.138832  149.469593  8.630603   0.154010   
4 1986-05-01  1986      5  0.135804   28.476741  8.451156   0.138832   

   ndvi_lag3  ndvi_lag12  precip_lag1  precip_lag3  temp_lag1  precip_roll_3m  \
0   0.183280    0.097327    92.110133    50.888336   8.403750      201.365890   
1   0.154364    0.122975    80.930177    28.325579   8.422828      268.069660   
2   

/tmp/ipython-input-2800052522.py:530: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = df_fore.groupby('period')['pred_ndvi'].describe().round(4)


--- Estadísticas de 5 Años (Huanico - DryWarmer) ---
           count    mean     std     min     25%     50%     75%     max
period                                                                  
2025-2030   72.0  0.1061  0.0130  0.0857  0.0975  0.1034  0.1170  0.1343
2031-2035   60.0  0.1114  0.0098  0.0960  0.1037  0.1094  0.1199  0.1270
2036-2040   60.0  0.1148  0.0072  0.1015  0.1081  0.1161  0.1208  0.1262
2041-2045   60.0  0.1150  0.0039  0.1054  0.1123  0.1154  0.1170  0.1234
2046-2050   60.0  0.1147  0.0019  0.1120  0.1124  0.1157  0.1160  0.1168
Estadísticas guardadas en: ndvi_results_v2/Huanico_DryWarmer_stats_5_anos.csv


Generando forecast escenario: WetterWarmer
Iniciando pronóstico iterativo con features complejos...


/tmp/ipython-input-2800052522.py:530: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = df_fore.groupby('period')['pred_ndvi'].describe().round(4)


--- Estadísticas de 5 Años (Huanico - WetterWarmer) ---
           count    mean     std     min     25%     50%     75%     max
period                                                                  
2025-2030   72.0  0.1029  0.0146  0.0841  0.0904  0.0988  0.1156  0.1354
2031-2035   60.0  0.1029  0.0144  0.0841  0.0911  0.0981  0.1155  0.1292
2036-2040   60.0  0.1036  0.0142  0.0836  0.0931  0.0981  0.1134  0.1317
2041-2045   60.0  0.1035  0.0142  0.0841  0.0938  0.0981  0.1134  0.1320
2046-2050   60.0  0.1040  0.0140  0.0863  0.0938  0.0999  0.1144  0.1317
Estadísticas guardadas en: ndvi_results_v2/Huanico_WetterWarmer_stats_5_anos.csv


Modelo y pronósticos (V2) almacenados en ndvi_results_v2

INICIANDO ANÁLISIS MEJORADO PARA: Michiquillay
(Datos desde 1985 hasta 2024)
Iniciando extracción para Michiquillay desde 1985 hasta 2024...


...

/tmp/ipython-input-2800052522.py:530: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = df_fore.groupby('period')['pred_ndvi'].describe().round(4)


--- Estadísticas de 5 Años (Michiquillay - DryWarmer) ---
           count    mean     std     min     25%     50%     75%     max
period                                                                  
2025-2030   72.0  0.0987  0.0249  0.0709  0.0807  0.0859  0.1191  0.1445
2031-2035   60.0  0.1100  0.0215  0.0795  0.0889  0.1068  0.1226  0.1547
2036-2040   60.0  0.1189  0.0201  0.0882  0.1049  0.1158  0.1411  0.1532
2041-2045   60.0  0.1391  0.0099  0.1032  0.1363  0.1424  0.1462  0.1492
2046-2050   60.0  0.1441  0.0033  0.1394  0.1398  0.1459  0.1468  0.1469
Estadísticas guardadas en: ndvi_results_v2/Michiquillay_DryWarmer_stats_5_anos.csv


Generando forecast escenario: WetterWarmer
Iniciando pronóstico iterativo con features complejos...


/tmp/ipython-input-2800052522.py:530: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = df_fore.groupby('period')['pred_ndvi'].describe().round(4)


--- Estadísticas de 5 Años (Michiquillay - WetterWarmer) ---
           count    mean     std     min     25%     50%     75%     max
period                                                                  
2025-2030   72.0  0.0959  0.0242  0.0700  0.0788  0.0832  0.1104  0.1515
2031-2035   60.0  0.0954  0.0229  0.0723  0.0794  0.0824  0.1086  0.1467
2036-2040   60.0  0.0951  0.0227  0.0735  0.0793  0.0813  0.1041  0.1461
2041-2045   60.0  0.0945  0.0227  0.0765  0.0794  0.0809  0.1010  0.1454
2046-2050   60.0  0.0935  0.0212  0.0776  0.0792  0.0807  0.0990  0.1410
Estadísticas guardadas en: ndvi_results_v2/Michiquillay_WetterWarmer_stats_5_anos.csv


Modelo y pronósticos (V2) almacenados en ndvi_results_v2

Proceso finalizado. Revisa la carpeta 'ndvi_results_v2' para CSVs, modelos y PNGs.
Usa el ícono de la carpeta a la izquierda para ver tus archivos.


/tmp/ipython-input-2800052522.py:530: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = df_fore.groupby('period')['pred_ndvi'].describe().round(4)
